# 🎬 Image → Video Pipeline (무료/저비용 프로토타입)

**구성:** Pollinations(이미지) → Real-ESRGAN(업스케일) → Gemini(모션 프롬프트) → **Wan 2.2 I2V(영상)** → RIFE(보간) → Real-ESRGAN(영상 업스케일) → FFmpeg(색보정/오디오)

**목적:** Wan 2.2 image-to-video 품질을 본인 워크플로우에서 검증. 만족스러우면 spec-kit으로 본격화.

---

### ⚙️ 실행 전 필수
1. **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU (T4 이상)** 설정
2. 무료 T4는 480~720p, Colab Pro의 L4/A100이면 720~1080p 가능
3. Google Drive 마운트로 중간 산출물 보존 (세션 끊김 대비)
4. Gemini API 키 (선택 — 모션 프롬프트 자동 생성용). 없으면 수동 프롬프트 사용

> ⏱ 전체 1회 실행 예상: 모델 다운로드 포함 첫 실행 20~40분, 이후 컷당 5~15분 (GPU에 따라)


## 0. GPU 확인 & Drive 마운트

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print("⚠️ GPU가 없습니다. 런타임 유형을 GPU로 바꿔주세요.")


In [ ]:
# Google Drive 마운트 (중간 산출물 보존)
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/img2video'
os.makedirs(f'{WORK}/01_images', exist_ok=True)
os.makedirs(f'{WORK}/02_upscaled', exist_ok=True)
os.makedirs(f'{WORK}/03_videos_raw', exist_ok=True)
os.makedirs(f'{WORK}/04_interpolated', exist_ok=True)
os.makedirs(f'{WORK}/05_upscaled_video', exist_ok=True)
os.makedirs(f'{WORK}/06_final', exist_ok=True)
os.makedirs(f'{WORK}/models', exist_ok=True)
print("작업 디렉토리:", WORK)


## 1. 설정 (여기만 바꾸면 됩니다)

포맷·길이·프롬프트 등 주요 파라미터를 여기서 조정합니다.

In [ ]:
# ===== 사용자 설정 =====
CONFIG = {
    # 이미지 생성
    "image_prompt": "a serene mountain lake at golden hour, cinematic, photorealistic, dramatic clouds reflecting on water",
    "image_model": "flux",          # flux | turbo | flux-realism
    "image_seed": 42,               # 재현성 (None이면 랜덤)

    # 출력 포맷
    "aspect": "16:9",               # "16:9" (가로) | "9:16" (숏폼) | "1:1"
    "base_width": 1280,             # 영상 생성 기준 (GPU 여유 따라 조정)

    # 영상 생성 (Wan 2.2 I2V)
    "motion_prompt": None,          # None이면 Gemini가 자동 생성. 직접 쓰려면 영어 문자열
    "video_length_frames": 49,      # Wan 2.2 권장 (약 3초 @ 16fps)
    "video_fps": 16,                # Wan 2.2 native fps
    "video_steps": 30,              # 추론 스텝 (높을수록 품질↑ 느림↑)
    "guidance_scale": 5.0,

    # 후처리
    "interpolate_to_fps": 48,       # RIFE 목표 fps (16→48 = 3배)
    "upscale_video": True,          # Real-ESRGAN 영상 업스케일 on/off
    "upscale_factor": 2,            # 2배 (1280→2560 등)

    # 색보정 (FFmpeg eq 필터)
    "contrast": 1.08,
    "saturation": 1.12,
    "brightness": 0.02,

    # 오디오 (선택)
    "bgm_path": None,               # f"{WORK}/audio/bgm.mp3" 형태, 없으면 무음
}

# 종횡비 → 해상도 매핑
_ASPECT = {
    "16:9": (CONFIG["base_width"], int(CONFIG["base_width"] * 9 / 16)),
    "9:16": (int(CONFIG["base_width"] * 9 / 16), CONFIG["base_width"]),
    "1:1":  (CONFIG["base_width"], CONFIG["base_width"]),
}
CONFIG["width"], CONFIG["height"] = _ASPECT[CONFIG["aspect"]]
# 8의 배수로 정렬 (모델 요구사항)
CONFIG["width"]  = (CONFIG["width"]  // 8) * 8
CONFIG["height"] = (CONFIG["height"] // 8) * 8

print("해상도:", CONFIG["width"], "x", CONFIG["height"])
print("영상 길이:", round(CONFIG["video_length_frames"] / CONFIG["video_fps"], 1), "초")


In [ ]:
# Gemini API 키 (선택 — 모션 프롬프트 자동 생성)
# Colab 좌측 🔑 Secrets에 GEMINI_API_KEY 추가하거나 아래에 직접 입력
GEMINI_API_KEY = ""
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ Secrets에서 Gemini 키 로드")
except Exception:
    if GEMINI_API_KEY:
        print("✅ 직접 입력한 Gemini 키 사용")
    else:
        print("⚠️ Gemini 키 없음 → 모션 프롬프트는 수동으로 작성하세요 (CONFIG['motion_prompt'])")


## 2. Pollinations 이미지 생성 (무료, API 키 불필요)

In [ ]:
import requests, urllib.parse, time
from pathlib import Path

def generate_pollinations_image(prompt, w, h, model="flux", seed=42):
    enc = urllib.parse.quote(prompt)
    url = (f"https://image.pollinations.ai/prompt/{enc}"
           f"?width={w}&height={h}&model={model}&nologo=true&enhance=true")
    if seed is not None:
        url += f"&seed={seed}"
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=180)
            if r.status_code == 200 and len(r.content) > 5000:
                return r.content
            print(f"  재시도 {attempt+1} (status={r.status_code})")
        except Exception as e:
            print(f"  재시도 {attempt+1} (error={e})")
        time.sleep(5)
    raise RuntimeError("Pollinations 이미지 생성 실패")

img_path = f"{WORK}/01_images/source.png"
content = generate_pollinations_image(
    CONFIG["image_prompt"], CONFIG["width"], CONFIG["height"],
    CONFIG["image_model"], CONFIG["image_seed"]
)
Path(img_path).write_bytes(content)
print("✅ 이미지 저장:", img_path)

from IPython.display import Image as IPImage, display
display(IPImage(img_path))


## 3. 모션 프롬프트 자동 생성 (Gemini Vision)

이미지를 분석해서 자연스러운 카메라/피사체 모션 프롬프트를 영어로 생성합니다.
Gemini 키가 없으면 이 셀을 건너뛰고 `CONFIG['motion_prompt']`를 직접 채우세요.

In [ ]:
motion_prompt = CONFIG.get("motion_prompt")

if not motion_prompt and GEMINI_API_KEY:
    !pip install -q google-genai
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=GEMINI_API_KEY)
    image_bytes = Path(img_path).read_bytes()
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
            ("Describe the most natural, cinematic motion to animate this still image "
             "as a short video. Focus on subtle, physically plausible camera movement "
             "and subject motion. Output ONLY a concise English motion prompt, no preamble."),
        ],
        config=types.GenerateContentConfig(temperature=0.4),
    )
    motion_prompt = resp.text.strip()
    print("🤖 자동 생성 모션 프롬프트:")
elif not motion_prompt:
    # 폴백 기본값
    motion_prompt = "slow cinematic camera push-in, gentle ripples on water, soft drifting clouds"
    print("📝 기본 모션 프롬프트 사용:")

print(motion_prompt)
CONFIG["motion_prompt"] = motion_prompt


## 4. (선택) 입력 이미지 업스케일 — Real-ESRGAN

Pollinations 이미지가 충분히 선명하면 건너뛰어도 됩니다.
영상 모델 입력 해상도에 맞추므로 보통은 생략 가능 — 기본 OFF.

In [ ]:
UPSCALE_INPUT = False  # 필요 시 True

if UPSCALE_INPUT:
    !pip install -q realesrgan basicsr
    # 간단 버전: realesrgan-ncnn-vulkan 바이너리 사용 권장 (아래 영상 업스케일 셀 참조)
    print("입력 이미지 업스케일은 보통 불필요. 영상 단계에서 처리합니다.")
else:
    print("⏭ 입력 업스케일 건너뜀")


## 5. ⭐ 핵심 — Wan 2.2 I2V 영상 생성

ComfyUI 대신 **diffusers 파이프라인**으로 간단히 구동합니다.
첫 실행 시 모델 가중치 다운로드(~수 GB)가 있어 시간이 걸립니다. Drive에 캐싱됩니다.

> VRAM 부족 시: `base_width`를 줄이거나(예 832), `video_length_frames`를 줄이세요(예 33).
> T4(16GB)는 480~640p, L4/A100은 720~1080p 권장.

In [ ]:
# Wan 2.2 I2V via diffusers
!pip install -q "diffusers>=0.31" transformers accelerate safetensors ftfy imageio imageio-ffmpeg

import torch
from diffusers import AutoModel  # 버전에 따라 WanImageToVideoPipeline 사용
from diffusers.utils import export_to_video, load_image

# 모델 캐시를 Drive로
import os
os.environ["HF_HOME"] = f"{WORK}/models/hf_cache"

print("⏳ Wan 2.2 I2V 파이프라인 로드 중... (첫 실행 시 다운로드)")

# 주: diffusers 버전에 따라 클래스명이 다를 수 있음.
# 최신 권장: WanImageToVideoPipeline
try:
    from diffusers import WanImageToVideoPipeline
    MODEL_ID = "Wan-AI/Wan2.2-I2V-A14B-Diffusers"  # 고품질(무거움)
    # T4 등 저사양이면 경량 변형으로 교체:
    # MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
    pipe = WanImageToVideoPipeline.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16
    )
    pipe.to("cuda")
    pipe.enable_model_cpu_offload()   # VRAM 절약
    HAVE_WAN = True
    print("✅ Wan 2.2 I2V 로드 완료:", MODEL_ID)
except Exception as e:
    HAVE_WAN = False
    print("⚠️ Wan 2.2 diffusers 로드 실패:", e)
    print("→ 아래 '대안: ComfyUI' 또는 '대안: 유료 API' 셀을 사용하세요.")


In [ ]:
# 영상 생성 실행
if HAVE_WAN:
    from diffusers.utils import load_image, export_to_video
    image = load_image(img_path)

    print("🎬 영상 생성 중... (GPU에 따라 수 분 소요)")
    result = pipe(
        image=image,
        prompt=CONFIG["motion_prompt"],
        negative_prompt="blurry, distorted, low quality, artifacts, warping",
        height=CONFIG["height"],
        width=CONFIG["width"],
        num_frames=CONFIG["video_length_frames"],
        num_inference_steps=CONFIG["video_steps"],
        guidance_scale=CONFIG["guidance_scale"],
    )
    frames = result.frames[0]

    raw_video = f"{WORK}/03_videos_raw/raw.mp4"
    export_to_video(frames, raw_video, fps=CONFIG["video_fps"])
    print("✅ 원본 영상 저장:", raw_video)

    from IPython.display import Video
    display(Video(raw_video, embed=True, width=480))


### 대안 A: ComfyUI로 Wan 2.2 (diffusers 실패/품질 부족 시)

diffusers 경로가 막히면 ComfyUI가 가장 안정적입니다. 아래는 설치 스텁입니다.

In [ ]:
# ComfyUI 설치 (필요 시에만 실행)
RUN_COMFYUI = False

if RUN_COMFYUI:
    %cd /content
    !git clone https://github.com/comfyanonymous/ComfyUI
    %cd ComfyUI
    !pip install -q -r requirements.txt
    # Wan 2.2 모델/워크플로우는 ComfyUI Manager로 설치 권장
    # 모델: Wan-AI/Wan2.2-I2V (HuggingFace) → models/diffusion_models/
    # cloudflared로 외부 접속 터널:
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
    print("ComfyUI를 실행하고 cloudflared 터널로 접속해 워크플로우를 구성하세요.")
    print("# !python main.py --listen & ./cloudflared tunnel --url http://localhost:8188")
else:
    print("⏭ ComfyUI 경로 미사용")


### 대안 B: 유료 API로 핵심 컷만 (하이브리드)

무료 경로 품질이 부족한 핵심 컷만 Veo/Kling으로. 키 있을 때만.

In [ ]:
# Veo 3.1 (Gemini API) — 하이브리드용
USE_PAID_API = False

if USE_PAID_API and GEMINI_API_KEY:
    from google import genai
    from google.genai import types
    import time
    client = genai.Client(api_key=GEMINI_API_KEY)
    op = client.models.generate_videos(
        model="veo-3.1-generate-preview",
        prompt=CONFIG["motion_prompt"],
        image=types.Image(image_bytes=Path(img_path).read_bytes(), mime_type="image/png"),
        config=types.GenerateVideosConfig(
            aspect_ratio=CONFIG["aspect"], number_of_videos=1,
            duration_seconds=8, resolution="1080p",
        ),
    )
    while not op.done:
        time.sleep(10); op = client.operations.get(op)
    vid = op.response.generated_videos[0]
    client.files.download(file=vid.video)
    raw_video = f"{WORK}/03_videos_raw/raw_veo.mp4"
    vid.video.save(raw_video)
    print("✅ Veo 영상 저장:", raw_video)
else:
    print("⏭ 유료 API 미사용")


## 6. 프레임 보간 — RIFE (부드러움 향상)

16fps → 48fps 등으로 보간하여 모션을 매끄럽게 만듭니다.

In [ ]:
# Practical-RIFE
%cd /content
import os
if not os.path.exists('/content/Practical-RIFE'):
    !git clone -q https://github.com/hzwer/Practical-RIFE
    %cd Practical-RIFE
    !pip install -q -r requirements.txt
    # 모델 가중치는 README의 안내대로 train_log/ 에 배치 필요
    print("⚠️ RIFE 모델 가중치(train_log)를 README 안내대로 다운로드하세요.")
    print("   https://github.com/hzwer/Practical-RIFE")

multiplier = max(1, round(CONFIG["interpolate_to_fps"] / CONFIG["video_fps"]))
interp_out = f"{WORK}/04_interpolated/interp.mp4"

# 가중치가 준비된 경우:
# %cd /content/Practical-RIFE
# !python inference_video.py --multi {multiplier} --video {raw_video} --output {interp_out}

print(f"보간 배수: {multiplier}x  (목표 {CONFIG['interpolate_to_fps']}fps)")
print("RIFE 가중치 준비 후 위 주석 해제하여 실행")

# 가중치 없으면 FFmpeg minterpolate로 폴백 (품질은 RIFE보다 낮음)
FALLBACK_FFMPEG_INTERP = True
if FALLBACK_FFMPEG_INTERP:
    !ffmpeg -y -i {raw_video} -vf "minterpolate=fps={CONFIG['interpolate_to_fps']}:mi_mode=mci:mc_mode=aobmc:vsbmc=1" {interp_out} 2>/dev/null
    print("✅ FFmpeg minterpolate 폴백 완료:", interp_out)


## 7. (선택) 영상 업스케일 — Real-ESRGAN

ncnn-vulkan 바이너리가 Colab GPU에서 빠릅니다. 프레임 분해 → 업스케일 → 재합성.

In [ ]:
current_video = interp_out  # 직전 단계 결과

if CONFIG["upscale_video"]:
    %cd /content
    if not os.path.exists('/content/realesrgan-ncnn-vulkan'):
        !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesrgan-ncnn-vulkan-20220424-ubuntu.zip
        !unzip -q -o realesrgan-ncnn-vulkan-20220424-ubuntu.zip -d realesrgan
        !chmod +x realesrgan/realesrgan-ncnn-vulkan

    import shutil
    frames_dir = "/content/_frames_in"; out_dir = "/content/_frames_out"
    shutil.rmtree(frames_dir, ignore_errors=True); shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(frames_dir); os.makedirs(out_dir)

    # 프레임 추출
    !ffmpeg -y -i {current_video} {frames_dir}/f_%05d.png 2>/dev/null
    # 업스케일 (realesrgan-x4plus, 출력 후 scale 조정)
    !./realesrgan/realesrgan-ncnn-vulkan -i {frames_dir} -o {out_dir} -n realesrgan-x4plus -s {CONFIG['upscale_factor']} 2>/dev/null
    # 재합성
    upscaled_video = f"{WORK}/05_upscaled_video/upscaled.mp4"
    !ffmpeg -y -framerate {CONFIG['interpolate_to_fps']} -i {out_dir}/f_%05d.png -c:v libx264 -crf 16 -pix_fmt yuv420p {upscaled_video} 2>/dev/null
    current_video = upscaled_video
    print("✅ 업스케일 완료:", upscaled_video)
else:
    print("⏭ 영상 업스케일 건너뜀")


## 8. 색보정 + 오디오 — FFmpeg (최종 출력)

색감을 시네마틱하게 다듬고, BGM이 있으면 합성합니다.

In [ ]:
final_video = f"{WORK}/06_final/final.mp4"
eq = (f"eq=contrast={CONFIG['contrast']}:"
      f"saturation={CONFIG['saturation']}:"
      f"brightness={CONFIG['brightness']}")
fade = "fade=in:0:8"

if CONFIG["bgm_path"] and os.path.exists(CONFIG["bgm_path"]):
    !ffmpeg -y -i {current_video} -i "{CONFIG['bgm_path']}"         -vf "{eq},{fade}" -c:v libx264 -crf 17 -preset slow         -c:a aac -b:a 192k -shortest {final_video} 2>/dev/null
else:
    !ffmpeg -y -i {current_video}         -vf "{eq},{fade}" -c:v libx264 -crf 17 -preset slow         -pix_fmt yuv420p {final_video} 2>/dev/null

print("🎉 최종 영상 저장:", final_video)
from IPython.display import Video
display(Video(final_video, embed=True, width=640))


## 9. 결과 다운로드 / 정리

최종 영상은 이미 Google Drive(`06_final/`)에 저장되어 있습니다.
브라우저로 바로 받으려면 아래 셀 실행.

In [ ]:
from google.colab import files
files.download(final_video)


---

## 📌 다음 단계 체크리스트

품질을 평가하면서 아래를 확인하세요:

- [ ] **Wan 2.2 모션이 자연스러운가?** (warping/jitter 없는지)
- [ ] **소스 이미지 디테일이 영상에서 유지되는가?**
- [ ] **보간(RIFE)으로 충분히 부드러운가?** FFmpeg 폴백이면 RIFE 가중치 받아 비교
- [ ] **업스케일 후 디테일 향상 vs 처리 시간** 트레이드오프 만족스러운가?
- [ ] **GPU(T4) 한계 해상도** 어디까지 되는지 (OOM 지점 파악)

### 품질이 부족하면
1. `video_steps` 30 → 40~50 상향
2. `MODEL_ID`를 A14B(고품질)로, Colab Pro L4/A100 사용
3. 핵심 컷만 **대안 B(Veo/Kling 유료 API)** 로 하이브리드

### 품질이 만족스러우면
→ 이 파이프라인을 **spec-kit 5종**으로 정식화 + 로컬 오케스트레이션(BullMQ)으로 배치 자동화

---
*프로토타입 v1.0 — Pollinations + Wan 2.2 + RIFE + Real-ESRGAN + FFmpeg*
